In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Доли пропущенных признаков для экспериментов
MISSING_RATES = [0.0, 0.2, 0.5, 0.9]

# Настройка промпта для California
prompt_config = {
        "task": "Predict whether house price is above median",
        "pos_label": "yes",
        "neg_label": "no",
        "entity": "House",
        "question": "Is this house price above median?"
}

openml_id = 44090

base_output_dir = "/content/drive/MyDrive/finetuned_qwen_missing_California"

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):

    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, val_df, test_df = split_dataset(df, target_name)

df.head(5)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,2.1827,26.0,4.521429,0.921429,305.0,2.178571,40.05,-122.10,0
1,3.0755,32.0,4.623068,0.983353,3868.0,4.599287,32.77,-117.06,0
2,1.8235,40.0,4.701149,1.126437,928.0,3.555556,37.75,-122.16,0
3,1.4625,37.0,4.247845,1.105603,1673.0,3.605603,33.99,-118.28,0
4,1.9063,13.0,3.453125,0.984375,286.0,4.468750,33.97,-118.16,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20634 entries, 0 to 20633
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20634 non-null  float64
 1   HouseAge    20634 non-null  float64
 2   AveRooms    20634 non-null  float64
 3   AveBedrms   20634 non-null  float64
 4   Population  20634 non-null  float64
 5   AveOccup    20634 non-null  float64
 6   Latitude    20634 non-null  float64
 7   Longitude   20634 non-null  float64
 8   price       20634 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 1.4 MB


In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 12380):
  no:  6190 — 50.0%
  yes:  6190 — 50.0%

val (всего: 4127):
  no:  2063 — 50.0%
  yes:  2064 — 50.0%

test (всего: 4127):
  no:  2064 — 50.0%
  yes:  2063 — 50.0%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template_with_missing(row, feature_names, target_name, missing_rate=0.0, seed=None):

    rng = np.random.default_rng(seed)

    # Определяем, какие признаки пропустить
    n_drop = int(len(feature_names) * missing_rate)
    dropped = set(rng.choice(feature_names, size=n_drop, replace=False)) if n_drop > 0 else set()

    template_parts = []

    for feature in feature_names:
        if feature in dropped:
            continue  # пропускаем признак

        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    return text

# Тест
test_row = train_df.iloc[0]
display(train_df.head(1))
for rate in MISSING_RATES:
    text_output = row_to_text_template_with_missing(
        row=test_row,
        feature_names=feature_names,
        target_name=target_name,
        missing_rate=rate
    )

    print(f"Missing Rate: {rate * 100:.0f}%")
    print(text_output)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
6451,2.3156,51.0,3.963918,1.015464,1173.0,3.023196,34.07,-117.76,0


Missing Rate: 0%
The value of MedInc is 2.32. The value of HouseAge is 51.00. The value of AveRooms is 3.96. The value of AveBedrms is 1.02. The value of Population is 1173.00. The value of AveOccup is 3.02. The value of Latitude is 34.07. The value of Longitude is -117.76.
Missing Rate: 20%
The value of MedInc is 2.32. The value of HouseAge is 51.00. The value of AveRooms is 3.96. The value of AveBedrms is 1.02. The value of Population is 1173.00. The value of Latitude is 34.07. The value of Longitude is -117.76.
Missing Rate: 50%
The value of HouseAge is 51.00. The value of AveBedrms is 1.02. The value of AveOccup is 3.02. The value of Longitude is -117.76.
Missing Rate: 90%
The value of AveBedrms is 1.02.


In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip().rstrip('.,!? ')
    pos, neg = prompt_config['pos_label'].lower(), prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos: return 1
    if response == neg: return 0

    # Начинается с метки
    if response.startswith(pos): return 1
    if response.startswith(neg): return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words: return 1
    if neg in words: return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'yes'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob if y_prob is not None else y_pred),
        "F1":       f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":   recall_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template_with_missing(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of MedInc is 2.68. The value of HouseAge is 28.00. The value of AveRooms is 4.21. The value of AveBedrms is 1.01. The value of Population is 2218.00. The value of AveOccup is 3.57. The value of Latitude is 34.07. The value of Longitude is -118.18.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device, # Explicitly force all model layers to the detected device (GPU if available)
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0, seed=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt_missing(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'no', Probability: 0.3775


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, seed=42):
    df_0 = train_df[train_df[target_name] == 0]
    df_1 = train_df[train_df[target_name] == 1]

    df_majority = df_0 if len(df_0) > len(df_1) else df_1
    df_minority = df_1 if len(df_0) > len(df_1) else df_0

    print(f"До балансировки:")
    print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
    print(f"{prompt_config['pos_label']}: {len(df_minority)}")

    df_minority_up = resample(df_minority, replace=True,
                              n_samples=len(df_majority), random_state=seed)

    train_df_balanced = pd.concat([df_majority, df_minority_up])
    train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\nПосле балансировки:")
    print(train_df_balanced[target_name].value_counts())

    return train_df_balanced

train_df_balanced = balance_dataset(train_df, target_name)

До балансировки:
no:  6190
yes: 6190

После балансировки:
price
0    6190
1    6190
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения

def create_training_dataset(df, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Dataset (missing={missing_rate:.0%})")):
        input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, idx)
        target = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts


In [ ]:
def finetune_model(missing_rate, train_df_balanced, feature_names, target_name,
                   prompt_config, tokenizer, base_model, device,
                   num_epochs=3, batch_size=16):
    """
    Полный цикл fine-tuning для заданного missing_rate.
    Возвращает путь к лучшему checkpoint.
    """
    output_dir = f"{base_output_dir}_missing{int(missing_rate * 100):03d}"
    print(f"  Fine-tuning: missing_rate = {missing_rate:.0%}")
    print(f"  Output dir : {output_dir}")

    # Датасет
    train_texts = create_training_dataset(
        train_df_balanced, feature_names, target_name,
        prompt_config, tokenizer, missing_rate=missing_rate)
    train_dataset = Dataset.from_dict({"text": train_texts})

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding=False)

    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True,
                                          remove_columns=train_dataset.column_names)

    # LoRA
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=50,
        save_strategy="epoch",
        optim="adamw_torch_fused",
        warmup_steps=50,
        max_grad_norm=1.0,
        weight_decay=0.01,
        report_to="none",
        dataloader_num_workers=4,
        group_by_length=True,
    )

    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    t0 = time.time()
    trainer.train()
    print(f"Обучение завершено за {(time.time()-t0)/60:.1f} мин")

    trainer.save_model(output_dir)

    # Возвращаем базовую модель без LoRA весов (важно для следующего эксперимента)
    model_lora = model_lora.unload()
    torch.cuda.empty_cache()

    return output_dir

In [ ]:
import glob
import re
from peft import PeftModel

def evaluate_checkpoints_on_val(output_dir, val_df, feature_names, target_name,
                                 prompt_config, tokenizer, base_model, device,
                                 eval_missing_rate=0.0):

    checkpoints = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
    )
    print(f"Найдено {len(checkpoints)} checkpoints в {output_dir}")

    best_auc, best_checkpoint = 0.0, None
    for cp in checkpoints:
        model_eval = PeftModel.from_pretrained(base_model, cp).eval()

        y_true, y_prob = [], []
        for _, row in tqdm(val_df.iterrows(), total=len(val_df),
                           desc=f"Val eval {cp.split('/')[-1]}"):
            prompt = create_prompt_missing(
                row, feature_names, target_name, prompt_config, tokenizer,
                missing_rate=eval_missing_rate)
            _, prob = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer, device)
            y_true.append(row[target_name])
            y_prob.append(prob)

        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        auc = roc_auc_score(y_true, y_prob)
        print(f"  {cp.split('/')[-1]}  ROC-AUC={auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_checkpoint = cp

        del model_eval
        torch.cuda.empty_cache()

    print(f"Лучший checkpoint: {best_checkpoint}  (ROC-AUC={best_auc:.4f})")
    return best_checkpoint

In [ ]:
def evaluate_on_test(best_checkpoint, test_df, feature_names, target_name,
                     prompt_config, tokenizer, base_model, device,
                     eval_missing_rate=0.0):
    model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint).eval()

    y_true, y_pred, y_prob = [], [], []
    t0 = time.time()

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test eval"):
        prompt = create_prompt_missing(
            row, feature_names, target_name, prompt_config, tokenizer,
            missing_rate=eval_missing_rate)
        response, prob = predict_single_with_prob(
            prompt, prompt_config, model_finetuned, tokenizer, device)
        y_true.append(row[target_name])
        y_pred.append(parse_prediction(response, prompt_config))
        y_prob.append(prob)

    elapsed = time.time() - t0
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    boot = bootstrap_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), n_iter=1000)

    print("Метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("Bootstrap (mean±std):")
    for k, v in boot.items():
        print(f"  {k}: {v}")

    del model_finetuned
    torch.cuda.empty_cache()

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": elapsed,
        "time_per_sample": elapsed / len(y_true),
        "best_checkpoint": best_checkpoint,
        "eval_missing_rate": eval_missing_rate,
    }

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

all_results = {}

start_time = time.time()

for missing_rate in MISSING_RATES:
    tag = f"missing_{int(missing_rate * 100):03d}pct"
    print(f"Эксперимент: missing_rate = {missing_rate:.0%}")

    # 7.1 Fine-tuning
    output_dir = finetune_model(
        missing_rate=missing_rate,
        train_df_balanced=train_df_balanced,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        num_epochs=10,
        batch_size=16,
    )

    # 7.2 Выбор лучшего checkpoint по val
    best_cp = evaluate_checkpoints_on_val(
        output_dir=output_dir,
        val_df=val_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    # 7.3 Оценка на test
    result = evaluate_on_test(
        best_checkpoint=best_cp,
        test_df=test_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    all_results[tag] = result

print(f"Эксперимент завершен за {(time.time()-start_time)/60:.1f} мин")

Mounted at /content/drive
Эксперимент: missing_rate = 0%
  Fine-tuning: missing_rate = 0%
  Output dir : /content/drive/MyDrive/finetuned_qwen_missing_California_missing000


Dataset (missing=0%):   0%|          | 0/12380 [00:00<?, ?it/s]

Map:   0%|          | 0/12380 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss
50,0.793823
100,0.286759
150,0.284796
200,0.280978
250,0.278901
300,0.278302
350,0.276232
400,0.277744
450,0.274585
500,0.273618


...

Val eval checkpoint-3870:   0%|          | 0/4127 [00:00<?, ?it/s]

  checkpoint-3870  ROC-AUC=0.6704
Лучший checkpoint: /content/drive/MyDrive/finetuned_qwen_missing_California_missing090/checkpoint-3096  (ROC-AUC=0.6785)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Test eval:   0%|          | 0/4127 [00:00<?, ?it/s]

Метрики:
  ROC-AUC: 0.6759
  F1: 0.5349
  Accuracy: 0.6089
  Precision: 0.6596
  Recall: 0.4498
Bootstrap (mean±std):
  ROC-AUC: 0.6758±0.0082
  F1: 0.5346±0.0101
  Accuracy: 0.6090±0.0078
  Precision: 0.6594±0.0126
  Recall: 0.4496±0.0109
Эксперимент завершен за 922.5 мин


In [ ]:
all_results

{'missing_000pct': {'metrics': {'ROC-AUC': np.float64(0.9483539578847693),
   'F1': 0.8792480115690527,
   'Accuracy': 0.8786043130603344,
   'Precision': 0.8744007670182167,
   'Recall': 0.8841492971400873},
  'bootstrap': {'ROC-AUC': '0.9482±0.0030',
   'F1': '0.8789±0.0053',
   'Accuracy': '0.8784±0.0050',
   'Precision': '0.8741±0.0073',
   'Recall': '0.8838±0.0070'},
  'time_total': 933.5676052570343,
  'time_per_sample': 0.22620974200558136,
  'best_checkpoint': '/content/drive/MyDrive/finetuned_qwen_missing_California_missing000/checkpoint-1548',
  'eval_missing_rate': 0.0},
 'missing_020pct': {'metrics': {'ROC-AUC': np.float64(0.9338661616446283),
   'F1': 0.8589327146171694,
   'Accuracy': 0.8526774897019627,
   'Precision': 0.8237650200267023,
   'Recall': 0.8972370334464372},
  'bootstrap': {'ROC-AUC': '0.9337±0.0035',
   'F1': '0.8586±0.0058',
   'Accuracy': '0.8524±0.0055',
   'Precision': '0.8234±0.0084',
   'Recall': '0.8969±0.0066'},
  'time_total': 944.7466361522675,
 

In [ ]:
print("\nBootstrap (mean±std):")
for tag, res in all_results.items():
    rate = tag.replace("missing_", "").replace("pct", "%").lstrip("0") or "0%"
    print(f"\n  missing={rate}")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")


Bootstrap (mean±std):

  missing=%
    ROC-AUC: 0.9482±0.0030
    F1: 0.8789±0.0053
    Accuracy: 0.8784±0.0050
    Precision: 0.8741±0.0073
    Recall: 0.8838±0.0070

  missing=20%
    ROC-AUC: 0.9337±0.0035
    F1: 0.8586±0.0058
    Accuracy: 0.8524±0.0055
    Precision: 0.8234±0.0084
    Recall: 0.8969±0.0066

  missing=50%
    ROC-AUC: 0.8606±0.0055
    F1: 0.7781±0.0071
    Accuracy: 0.7729±0.0066
    Precision: 0.7597±0.0094
    Recall: 0.7976±0.0090

  missing=90%
    ROC-AUC: 0.6758±0.0082
    F1: 0.5346±0.0101
    Accuracy: 0.6090±0.0078
    Precision: 0.6594±0.0126
    Recall: 0.4496±0.0109


In [ ]:
from google.colab import runtime
runtime.unassign()

ROC-AUC (0.95): Высочайший уровень производительности. Модель демонстрирует исключительную способность классифицировать стоимость жилья. При 20% пропусков показатель удерживается на уровне 0.93, что подтверждает высокую информативность оставшихся географических и экономических признаков.

Recall (0.88): Высокая полнота предсказаний. Модель стабильно идентифицирует объекты недвижимости целевой категории. При 20% пропусков наблюдается парадоксальный рост полноты до 0.90, что говорит о смещении модели в сторону более уверенного охвата при небольшом дефиците данных.

Precision (0.87): Высокая точность, значительно превышающая показатели в задачах Bank и Blood. Это указывает на то, что зависимости в датасете California более линейны и понятны для модели Qwen. При 90% пропусков точность снижается до 0.66, но остается выше случайного угадывания.

Accuracy (0.88): Стабильная общая точность. Модель эффективно справляется с предсказанием медианной стоимости. При увеличении пропусков до 50% точность сохраняется на уровне 0.77, демонстрируя надежность мультизадачного дообучения для задач регрессионного типа, преобразованных в классификацию.

Время эксперимента: ~ 15 ч 18 мин.

Использовано оперативная памяти графического процесса: 33.8/80GB.

Графический процессор A100 80GB.